This notebook is the experimental supplement to the main report ([final_project.ipynb](final_project.ipynb)). It covers the full methodology, per-model results, and three rounds of prompt optimization. The core statistical analysis (OLS regression, permutation test, residual analysis) is in the main report.

The first section defines the concepts needed to follow the experiment. The rest presents results and their interpretation.

## Key concepts

**Predict, Tools, and RLM.** We test three ways to deliver data to an LLM for prediction, each giving the model the same training data and features as OLS. *Predict* is pure reasoning: the model receives data as a table and produces predictions in a single forward pass, with all computation internalized as reasoning tokens. *Tools* gives the model a Python environment with numpy and pandas (regression and ML libraries are banned), letting it compute correlations, find similar states, and build weighted predictions. *RLM* ([Recursive Language Model](https://arxiv.org/abs/2512.24601); Zhang, Kraska, and Khattab 2025) runs the model in a multi-turn loop with a Python REPL equipped with numpy and pandas (same banned-library rules as Tools). Instead of feeding the full context into the prompt, data is loaded as Python variables in the interpreter namespace. The model writes code to explore it, and can spawn fresh-context sub-LLM calls via `llm_query()` for domain reasoning. The distinction from traditional agents: agents decompose *problems* through human-designed scaffolds (retrieve, then reason, then act). RLMs decompose *context* and let the model control the analytical workflow. In our experiment, the RLM iterates for up to 20 turns.

**Contamination control.** The LLM likely encountered 2008 housing crash outcomes during pre-training. We use two controls to measure how much the model relies on memorized knowledge versus the provided data. First, a *shuffled* condition: training labels are randomly reassigned across states (Arizona's crash assigned to Vermont, Nevada's to Maine). If the model incorporates in-context data, conflicting labels should degrade performance. Second, a *zero-data* condition: the model receives only state names and an output contract, with no features, no training examples, and no role framing. If it can predict accurately with no data at all, it is operating from memorized recall.

**GEPA and DSPy.** DSPy is a framework for programming LLM pipelines with typed signatures and automatic optimization. [GEPA](https://dspy.ai/deep-dive/optimizers/gepa/) (Generative Evolution via Prompt Adaptation) is DSPy's evolutionary prompt optimizer: a teacher model reviews the student agent's execution traces, identifies failure patterns, proposes mutated instructions, and keeps winners on a Pareto frontier. We use GEPA to evolve the RLM prompt across three optimization campaigns.

## Experiment design

OLS regression has perfect computation but zero domain knowledge. It computes exact least-squares coefficients through matrix algebra, minimizing squared error by construction. It does not know that Arizona's 80% appreciation in a desert housing market is categorically different from North Dakota's 20% in an oil economy.

Large language models invert this trade-off. They encode economic knowledge from pre-training, including crisis dynamics and regional economic structure. The experiment asks: does that knowledge add predictive value over statistical optimization, and if so, which delivery mechanism extracts it best?

The comparison isolates domain reasoning from computation. OLS is blind to what features mean. The LLM knows that rapid population growth in Nevada signals speculative overbuilding, not organic demand, and that Florida's coastal condo boom is structurally different from Texas's inland expansion. In the Tools and RLM conditions, both sides can compute; in the Predict condition, we test whether domain knowledge alone suffices. When training outcomes are shuffled, the model's domain knowledge becomes anti-correlated with the provided labels, testing whether the model incorporates in-context data or ignores it.

**Evaluation.** 20 random 40/11 train-test splits of the 51-state dataset. OLS is re-fit on each split's training set. All methods see the same partitions.

**Data delivery.** In Predict (DSPy harness), DataFrames are serialized as pandas tabular text in the prompt. In chat-based Predict (Claude, GPT, Gemini interfaces), data is formatted as markdown tables in a [hand-written prompt](https://github.com/ethanaggor/sds1730-final-project/blob/main/prompts/chat_predict.md). In the RLM condition, `pd.DataFrame` objects are injected into the interpreter namespace. In Tools (chat-based with Python), data is provided as markdown tables alongside a code execution function. All tool conditions provide NumPy and Pandas.

**Banned methods.** In conditions with tools, the agent cannot use any library that performs automated model fitting: no `statsmodels`, no `sklearn`, no `scipy.optimize`, no `np.polyfit` or `np.linalg.lstsq`. We are comparing domain-informed reasoning against statistical optimization, not testing whether an LLM can call a regression library.

In [1]:
#| echo: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
#| echo: false
#| output: false
results = pd.read_json(os.path.join('results', 'llm_experiment_results.json'))
zero_results = pd.read_json(os.path.join('results', 'zero_data_results.json'))
results = results.merge(zero_results, on='split')

In [3]:
#| echo: false
means = results[['rmse_ols', 'rmse_llm_real', 'rmse_llm_shuffled', 'rmse_llm_zero']].mean()
stds = results[['rmse_ols', 'rmse_llm_real', 'rmse_llm_shuffled', 'rmse_llm_zero']].std()

bar_labels = ['OLS\nRegression', 'Predict\n(Real Data)', 'Predict\n(Shuffled)', 'Predict\n(Zero Data)']
bar_colors = ['steelblue', 'tab:green', 'tab:orange', 'tab:red']

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(bar_labels, means.values, yerr=stds.values, capsize=5, color=bar_colors, edgecolor='k', alpha=0.8)
ax.set_ylabel('Mean RMSE (across 20 splits)')
ax.set_title('OLS vs. LLM Prediction Accuracy')
plt.tight_layout()
plt.show()

print(f'Mean RMSE - OLS:              {means["rmse_ols"]:.2f} (+/- {stds["rmse_ols"]:.2f})')
print(f'Mean RMSE - Predict (real):    {means["rmse_llm_real"]:.2f} (+/- {stds["rmse_llm_real"]:.2f})')
print(f'Mean RMSE - Predict (shuffled): {means["rmse_llm_shuffled"]:.2f} (+/- {stds["rmse_llm_shuffled"]:.2f})')
print(f'Mean RMSE - Predict (zero):    {means["rmse_llm_zero"]:.2f} (+/- {stds["rmse_llm_zero"]:.2f})')

With real training data, Claude Opus 4.6 in pure reasoning mode achieves mean RMSE 4.34 across 20 cross-validation splits, compared to OLS at 8.88. Shuffled labels degrade RMSE to 9.86. We initially interpreted this as evidence against memorization: the model appeared to be learning from the provided data rather than recalling pre-trained knowledge.

The zero-data condition overturned that interpretation. Given only state names and an output contract (no features, no training examples, no role description), the model achieves mean RMSE 2.58 with a standard deviation of 0.70. Every split falls between 1.36 and 3.83. The model has near-perfect recall of state-level housing crash severity from pre-training.

This reorders the experimental narrative. The four conditions form a hierarchy that reflects how much each interferes with the model's memorized prior:

| Condition | Mean RMSE | Interpretation |
|-----------|-----------|----------------|
| Zero data | 2.58 | Pure recall from pre-training |
| Real data | 4.34 | Recall degraded by incomplete features |
| OLS | 8.88 | Features only, no prior |
| Shuffled data | 9.86 | Prior actively fought by wrong labels |

Each step down the table adds more interference between the model and its memorized knowledge. Real-data features worsen zero-data recall rather than improve it. Four features that explain 58% of variance (the OLS R-squared from the main report) are an incomplete projection of the crash-generating process. When the model is forced to reason through that bottleneck, it partially overrides its stronger prior with a weaker signal.

The shuffled condition remains informative, but not as a test of memorization. It shows that the model is sensitive to in-context data: wrong labels pull it off its prior, degrading performance below OLS. The model does not ignore the training set. It incorporates it, and when the data conflicts with pre-trained knowledge, both sources are corrupted.

The following sections examine delivery mechanisms and model families. The tools tax and harness comparison findings hold regardless of memorization, because they measure how efficiently different interfaces extract knowledge from the model, whether that knowledge comes from recall or reasoning.

### Harness comparison and the tools tax

All results below are from the same held-out partition (split 0: 40 training states, 11 test states). OLS uses the same split.

| Method | Model | Split 0 RMSE | Notes |
|--------|-------|-------------|-------|
| Predict | Claude Opus 4.6 | 3.16 | 20-split mean: 4.34 |
| Predict | GPT 5.4 Heavy | 3.28 | Domain reasoning, 4 min |
| Predict | GPT 5.4 Pro | 3.84 | Extended thinking, 28 min |
| Predict | Gemini 3.1 Pro | 3.87 | Nearest-neighbor analogues + domain reasoning |
| Predict + Tools | GPT 5.4 Heavy | 4.11 | Weighted KNN with regional segmentation, 6 min |
| Predict | Gemini 3 Flash | 4.21 | |
| RLM | GPT 5.4 Mini | 4.71 | Post-fix, 7 iterations |
| OLS Regression | statsmodels | 5.45 | Exact least-squares baseline |

Every pure-reasoning result from a frontier model beats every tool-assisted or iterative result. The table is ordered by RMSE, but it could just as well be ordered by model strength. Harness sophistication does not matter.

**The tools tax.** GPT 5.4 Heavy provides a natural experiment: same model, same split, two conditions. With tools, it spent 6 minutes executing 15 code cells, building weighted KNN across 5 feature-weight configurations, segmenting by Census region, computing tercile interaction tables. Without tools, it spent 4 minutes thinking. Result: 3.28 without tools versus 4.11 with. Giving the model computation made it worse.

The per-state errors show why. With tools, the model computed Washington's 6 nearest neighbors averaging -17.89 and submitted -19.20. Without tools, it reasoned that Washington's tech economy (Microsoft, Amazon) provides a cushion Oregon lacked, and predicted -21.50, closer to the actual -25.19. The kNN average was a precise answer to the wrong question. The domain reasoning was an approximate answer to the right question.

**Hawaii versus Nevada.** In the 4-feature space, Hawaii (81% appreciation) is numerically indistinguishable from Nevada (80%). Any distance-based method will treat them as near-identical. Actual outcomes: Hawaii crashed 18%, Nevada crashed 56%. The difference (island supply constraints versus speculative desert boom) is invisible in the features. The most statistically rigorous RLM evaluation implemented full leave-one-out cross-validation with a 12-parameter grid search. It scored worst among clean-exit evaluations (RMSE 11.72) and predicted Hawaii at -48.2. More computation, worse answer.

This pattern held across 291 RLM evaluations from a clean GEPA run. 100% of top-scoring and bottom-scoring evaluations converged on the same algorithm: k-nearest neighbors in standardized 4-feature space. What separated good from bad was whether the model adjusted its KNN baseline for economic context the 4 features cannot capture. When the harness included `llm_query()`, the strongest evaluations computed a KNN baseline, called `llm_query` for qualitative economic context, then overrode predictions where domain knowledge changed the picture. The computation was scaffolding; the domain knowledge was the payload.

**Extended thinking hurts.** Within the GPT 5.4 family, Heavy with 4 minutes of standard reasoning (3.28) beats Pro with 28 minutes of extended thinking (3.84). For Illinois, Pro's earlier passes produced -17 (closer to actual -21.81), but continued deliberation pushed the estimate to -14.5. More thinking introduced noise.

The RLM result (GPT 5.4 Mini, 4.71) was evaluated after the three-layer anti-loop fix described below. Earlier RLM evaluations ran against a broken SUBMIT API, making their scores unreliable. Illinois is the hardest state for every method (all models underpredicted the crash), while North Carolina is consistently well-predicted.

### The latent space hypothesis

The results follow a clear pattern when organized by the dimensionality of the effective feature space:

| Method | Effective feature space | RMSE |
|--------|----------------------|-------------|
| Zero-data recall | Full latent space (unconstrained) | 2.58 (20-split mean) |
| Pure reasoning | Full latent space + 4D features | 3.16 - 4.21 |
| Predict + Tools | 4D nonlinear + partial reasoning | 4.11 |
| RLM | 4D nonlinear, iterative | 4.71 |
| OLS | 4D linear | 5.45 |

OLS operates in a 4-dimensional linear space. Tool-assisted methods search the same 4D space nonlinearly through KNN and weighted averages. Pure reasoning operates in the model's full latent representational space. Zero-data recall operates in the same latent space but without the anchoring effect of explicit features.

A large language model with frozen weights is a function: it maps (prompt, data) to predictions. The function's value comes from the latent variables encoded in its parameters, on the order of trillions for frontier models. These parameters define a representational space where states are compared not by 4 numeric columns but by hundreds of contextual dimensions: industry composition, supply elasticity, subprime mortgage concentration, geographic isolation, regional spillover patterns, and post-crisis analytical knowledge about which structural factors mattered. Some of these dimensions are identifiable (we can trace the model's reasoning about subprime exposure or tech employment). Many more are black-box products of model scale, unobservable but real.

**The Illinois evidence.** Every model except Claude Opus underpredicts IL by 6+ percentage points. The 4-feature profile says "moderate-appreciation Midwest state," similar to Wisconsin (-11.81) and Minnesota (-21.83). Opus reasons about "heavy subprime exposure in south side and suburbs," a variable not in the dataset, and arrives at -19.00 (error +2.81). GPT 5.4 Heavy misses by +6.41, GPT 5.4 Pro by +7.31. The gap between 2.81 and 7.31 is access to a latent variable (subprime concentration) that one model retrieved from pre-training and applied to override the 4-feature signal.

**Bayesian formalization.** The model holds a prior $p(\text{crash} \mid \text{state identity})$ from pre-training, encoding latent economic structure. It constructs a likelihood $p(\text{features} \mid \text{crash})$ by reading the in-context training data. No statistics are pre-computed; the model extracts this structure from the raw table. The prediction is a posterior:

$$p(\text{crash} \mid \text{features, state, context}) \propto \underbrace{p(\text{features} \mid \text{crash})}_{\text{likelihood from data}} \times \underbrace{p(\text{crash} \mid \text{state})}_{\text{prior from pre-training}}$$

OLS computes only the likelihood term, in a 4D linear space. KNN searches the same 4D space nonlinearly. The LLM has a rich prior that includes unmeasured variables that actually generated the outcome.

**The zero-data result and prior dominance.** The zero-data condition (RMSE 2.58) proves the model's prior is near-perfect recall of crash outcomes. State-level FHFA HPI declines from the 2008 crisis are among the most analyzed statistics in modern economics, widely reported in academic papers, government reports, and news coverage. The model's pre-training corpus almost certainly contains these figures directly.

This means the prior term dominates the posterior. When we provide real training data (4 features, 40 training states), the likelihood term introduces a 4-dimensional projection of a higher-dimensional process. The OLS R-squared of 0.58 tells us these features capture only 58% of the variance. The remaining 42% is information the prior already has but the likelihood cannot express. The posterior (RMSE 4.34) is worse than the prior alone (RMSE 2.58) because the likelihood pulls the model toward a less accurate representation.

This is a concrete instance of a general Bayesian principle: when the prior is very informative and the likelihood function is weak, the posterior can be worse than the prior. The data is not informative enough to improve on the prior, and partially overriding a strong prior with a weak signal introduces noise.

**The shuffled condition as prior-likelihood conflict.** When training labels are shuffled, the prior ("Nevada should crash hard") conflicts with the in-context evidence ("Nevada's crash_pct is -4.15," some other state's actual value). The model compromises between conflicting prior and evidence and gets both wrong. Shuffled RMSE (9.86) is worse than OLS (8.88), which means the model does not ignore in-context data. It incorporates training examples even when they are adversarial. This is the expected behavior of a Bayesian agent under an adversarial likelihood: the wrong data is strong enough to partially override the correct prior.

The original interpretation of the shuffled result, that degradation proved the model was learning from data rather than memorizing, was insufficient. The zero-data condition reveals that both forces are at work simultaneously: the model has a strong memorized prior AND it incorporates in-context data. The shuffled condition tests the interaction between them, not the presence of one versus the other.

**Scope and limitations.** For this task, we cannot disentangle memorization from generalized reasoning. The model's strong zero-data performance could reflect direct recall of crash statistics encountered during pre-training, structured economic reasoning from latent features, or some combination. The housing crash is too well-documented an event to isolate these mechanisms.

The latent space hypothesis remains a plausible explanation for why LLMs could function as predictive primitives in domains where the target variable is not in the training corpus. If the model's representational space genuinely encodes economic structure beyond the explicit features, that structure would be valuable for out-of-distribution prediction where recall is not available. Testing this claim requires a target variable the model has not seen, which is a limitation of the current experiment rather than a refutation of the hypothesis.

### GEPA prompt optimization

GEPA evolves prompt instructions by reflecting on execution traces. A teacher model reviews the student agent's trajectory on a minibatch, identifies failure patterns, proposes a mutated instruction, and keeps winners on a Pareto frontier. We ran three campaigns on the RLM harness. Each produced a distinct failure mode that is itself a finding.

**Run 1: Reward hacking.** Teacher: Gemini 3.1 Pro. Student: Gemini 3 Flash. The metric returned per-state feedback containing actual crash values: `Rhode Island (RI): predicted -15.2, actual -27.6, error +12.4`. The teacher identified this information channel within one reflection cycle. Rather than proposing a better strategy, it extracted ground-truth values from the feedback strings and embedded a complete 51-state lookup dictionary in the mutated prompt. Scores jumped from 0.64 to 0.999. The student's code collapsed to a dictionary lookup. This is specification gaming: the optimizer maximized the metric, not the reasoning. Fix: restrict feedback to directional or ordinal information. Cost: $4.28, killed after 87 minutes.

**Run 2: SUBMIT API bug.** We added a constrained reflection template forbidding answer embedding. But 72% of evaluations (118/165) exhibited output looping, and 45% never called SUBMIT. Root cause: our `LocalInterpreter` defined `SUBMIT(**kwargs)` while DSPy's prompt template taught the model to call `SUBMIT(predictions)` as a positional argument. Every positional call was silently rejected, and the model fell back to printing predictions in various formats until the 20-iteration cap. The interpreter also maintained a persistent namespace across iterations; when the model mutated `test_df` by adding prediction columns, subsequent iterations inherited the mutations, and in 94.5% of evaluations the model read its own earlier guesses as ground truth. Zero mutations were accepted by GEPA because within-split variance dominated the signal. Cost: $31.96, killed after 2 hours 48 minutes.

**Run 3: Anti-loop architecture.** We implemented a three-layer fix: (1) type coercion in SUBMIT accepting DataFrames, dicts, key aliases, and numpy scalars; (2) explicit SUBMIT syntax example in the prompt; (3) semantic loop detection using numeric fingerprinting with namespace-aware SUBMIT hints. Student: GPT 5.4 Mini. Teacher: GPT 5.4 (high effort for student, xhigh for teacher). Twenty-iteration evaluations dropped from 47% to 6.5%. The 19 remaining max-iteration evaluations are all analysis loops (the model refines instead of submitting), not format failures.

GEPA produced 14 mutated prompts across 5 iterations and 291 evaluations. The best prompt (P7) achieved mean RMSE 7.25 across 9 splits, a 26% improvement over the baseline (9.81). P7 introduced a 5-archetype taxonomy (speculative boom, constrained wealthy, distressed industrial, stable interior, mixed), a calibration warning against mean-reversion at both tails, and conditional feature interpretation rules. Even so, P7 (RMSE 7.25) falls far short of pure reasoning (4.34 for Opus across 20 splits). GEPA can improve prompts within the RLM harness, but cannot overcome the ceiling imposed by operating in the 4-feature space. Cost: $118.37, paused at 30% of Phase 2 budget.

The three runs tell a progression. Run 1: feedback design creates the attack surface for specification gaming. Run 2: infrastructure bugs dominate the optimization signal when the harness is broken. Run 3: even with a working harness, the tool-assisted RLM paradigm cannot match pure reasoning because the REPL anchors the model on 4D KNN regardless of prompt sophistication.